# RLHF와 DPO 선호최적화 실습 - DPO Loss on Paired Preferences

- Tutorial ID: `mod-rlhf-dpo-lab`
- Tutorial: RLHF와 DPO 선호최적화 실습
- Section ID: `dpo-1`
- Section: DPO Loss on Paired Preferences


In [ ]:
# ============================================================
# 코드 읽는 법 — DPO Loss on Paired Preferences
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 선호쌍 chosen/rejected가 loss와 policy update 신호로 바뀌는 흐름 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
# log prob under policy and reference for chosen/rejected completions
pi_chosen = np.array([-1.2, -0.7, -2.0])
pi_reject = np.array([-1.8, -1.1, -1.6])
ref_chosen = np.array([-1.0, -0.9, -1.8])
ref_reject = np.array([-1.4, -1.0, -1.5])
beta = 0.2
margin = beta * ((pi_chosen-pi_reject) - (ref_chosen-ref_reject))
loss = -np.log(1/(1+np.exp(-margin)))
print('DPO margins:', np.round(margin, 3))
print('pair losses:', np.round(loss, 3))
print('mean loss:', round(loss.mean(), 3))
print('preference accuracy:', np.mean(margin > 0))